In [1]:
from google.colab import files
uploaded = files.upload()

Saving spam.csv to spam.csv


In [2]:
import pandas as pd
df = pd.read_csv("spam.csv",encoding="latin-1")

print(df.head())
print(df.tail())

     v1                                                 v2 Unnamed: 2  \
0   ham  Go until jurong point, crazy.. Available only ...        NaN   
1   ham                      Ok lar... Joking wif u oni...        NaN   
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...        NaN   
3   ham  U dun say so early hor... U c already then say...        NaN   
4   ham  Nah I don't think he goes to usf, he lives aro...        NaN   

  Unnamed: 3 Unnamed: 4  
0        NaN        NaN  
1        NaN        NaN  
2        NaN        NaN  
3        NaN        NaN  
4        NaN        NaN  
        v1                                                 v2 Unnamed: 2  \
5567  spam  This is the 2nd time we have tried 2 contact u...        NaN   
5568   ham              Will Ì_ b going to esplanade fr home?        NaN   
5569   ham  Pity, * was in mood for that. So...any other s...        NaN   
5570   ham  The guy did some bitching but I acted like i'd...        NaN   
5571   ham               

In [3]:
print (df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB
None


In [4]:
print(df.columns)

Index(['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


In [5]:
print(df.isnull().sum())

v1               0
v2               0
Unnamed: 2    5522
Unnamed: 3    5560
Unnamed: 4    5566
dtype: int64


In [6]:
df=df.dropna()
print(df.isnull().sum())

v1            0
v2            0
Unnamed: 2    0
Unnamed: 3    0
Unnamed: 4    0
dtype: int64


In [7]:
df=df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'])
print(df.columns)

Index(['v1', 'v2'], dtype='object')


In [8]:
df.head()

,v1,v2
281,ham,\Wen u miss someone
1038,ham,"Edison has rightly said, \A fool can ask more ..."
2255,ham,I just lov this line: \Hurt me with the truth
3525,ham,\HEY BABE! FAR 2 SPUN-OUT 2 SPK AT DA MO... DE...
4668,ham,"When I was born, GOD said, \Oh No! Another IDI..."


In [9]:
df.drop_duplicates(inplace=True)

In [38]:

from sklearn.feature_extraction.text import TfidfVectorizer

df = pd.read_csv('spam.csv', encoding='latin-1')

df['v1'] = df['v1'].astype(str).str.strip().str.lower()
df['v1'] = df['v1'].map({'ham': 0, 'spam': 1})

vectorizer = TfidfVectorizer(max_features=3000, stop_words='english')
X = vectorizer.fit_transform(df['v2']).toarray()


print("Umbo jipya la X:", X.shape)

Umbo jipya la X: (5572, 3000)


In [39]:
y=df["v1"].values

In [40]:
import torch

x_tensor=torch.tensor(X,dtype=torch.float32)
y_tensor=torch.tensor(y,dtype=torch.float32).unsqueeze(1)

In [41]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x_tensor,y_tensor,test_size=0.2,random_state=42)

In [42]:
x_train.shape,x_test.shape,y_train.shape,y_test.shape

(torch.Size([4457, 3000]),
 torch.Size([1115, 3000]),
 torch.Size([4457, 1]),
 torch.Size([1115, 1]))

In [43]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

In [44]:
class SpamClassfier(nn.Module):
  def __init__(self):
    super(SpamClassfier,self).__init__()

    self.fc1=nn.Linear(3000,128)
    self.fc2=nn.Linear(128,64)
    self.fc3=nn.Linear(64,1)

  def forward(self,x):
    x=F.relu(self.fc1(x))
    x=F.relu(self.fc2(x))
    x=torch.sigmoid(self.fc3(x))
    return x

model=SpamClassfier()
print(model)

SpamClassfier(
  (fc1): Linear(in_features=3000, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)


In [45]:
criterion=nn.BCELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)

epochs=10

for epoch in range(epochs):
  model.train()

  optimizer.zero_grad()

  prediction=model(x_train)

  loss=criterion(prediction,y_train)
  print(loss)

  loss.backward()

  optimizer.step()

  predicted_clases=(prediction>=0.5).float()

  correct=(predicted_clases== y_train).sum().item()

  accuracy=(correct/y_train.shape[0])*100

  if (epoch + 1) % 2 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss.item():.4f} | Accuracy: {accuracy:.2f}%")

tensor(0.7097, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [1/10] | Loss: 0.7097 | Accuracy: 13.39%
tensor(0.7063, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [2/10] | Loss: 0.7063 | Accuracy: 13.39%
tensor(0.7027, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6990, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [4/10] | Loss: 0.6990 | Accuracy: 13.39%
tensor(0.6951, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6908, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [6/10] | Loss: 0.6908 | Accuracy: 69.51%
tensor(0.6863, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6814, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [8/10] | Loss: 0.6814 | Accuracy: 97.64%
tensor(0.6761, grad_fn=<BinaryCrossEntropyBackward0>)
tensor(0.6705, grad_fn=<BinaryCrossEntropyBackward0>)
Epoch [10/10] | Loss: 0.6705 | Accuracy: 94.73%


In [46]:
model.eval()

with torch.no_grad():
  test_prediction=model(x_test)
  test_loss=criterion(test_prediction,y_test)

  test_predicted=(test_prediction>=0.5).float()
  correct_test=(test_predicted==y_test).sum().item()
  test_accuracy=(correct_test/y_test.shape[0])*100

print(f"Test Loss: {test_loss.item():.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.6657
Test Accuracy: 92.20%


In [47]:
def predict_spam(text):

  text_vectorized = vectorizer.transform([text]).toarray()


  text_tensor = torch.tensor(text_vectorized, dtype=torch.float32)


  model.eval()

  with torch.no_grad():

    prediction = model(text_tensor)


    predicted_class = (prediction >= 0.5).float()


    if predicted_class.item() == 1:
      return "spam"
    else:
      return "ham"


example_text_1 = "Free entry in 2 a wkly comp to win FA Cup final tickets 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's"
example_text_2 = "Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030"
example_text_3 = "Nah I don't think he goes to usf, he lives around here though"

print(f"'{example_text_1}' is: {predict_spam(example_text_1)}")
print(f"'{example_text_2}' is: {predict_spam(example_text_2)}")
print(f"'{example_text_3}' is: {predict_spam(example_text_3)}")

'Free entry in 2 a wkly comp to win FA Cup final tickets 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's' is: spam
'Had your mobile 11 months or more? U R entitled to Update to the latest colour mobiles with camera for Free! Call The Mobile Update Co FREE on 08002986030' is: spam
'Nah I don't think he goes to usf, he lives around here though' is: ham
